In [1]:
%pip install numpy==2.3.5 scipy==1.17.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 54.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 39.4 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.6
    Uninstalling numpy-2.4.6:
      Successfully uninstalled numpy-2.4.6
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [scipy]


In [3]:
import numpy as np
from scipy.optimize import minimize

names = ["소고기","버섯","양파","마늘","새우","버터","빵","와인"]
prices = [9000,3000,1000,500,4000,2000,1500,5000]
preferences = {"청소년":[3,4,3,3,4,2,4,1],"성인":[4,3,3,3,3,4,2,3],"시니어":[2,3,4,4,3,2,3,1]}
pairs = [[0,1,6],[0,5,3],[0,7,3],[1,2,2],[2,3,1],[3,4,2],[4,5,3],[5,6,1]]
group = "성인"
budget = 12000

# 1. 0과 1로 된 모든 비트열 만들기
n = len(names)
index = np.arange(2**n)
bits = (index[:, None] >> np.arange(n)) & 1

# 2. 업무 점수와 제약조건 계산
preference = np.array(preferences[group])
price = np.array(prices)
bonus = np.zeros(len(bits), dtype=int)
for i, j, points in pairs:
    bonus += points * bits[:, i] * bits[:, j]
score = bits @ preference + bonus
spending = bits @ price
count = bits.sum(axis=1)
feasible = (count == 2) & (spending <= budget)

# 3. QUBO 비용과 Ising 에너지 계산
penalty = int(preference.sum() + sum(points for _, _, points in pairs) + 1)
energy = -score + penalty * (count - 2)**2
for i in range(n):
    for j in range(i + 1, n):
        if prices[i] + prices[j] > budget:
            energy += penalty * bits[:, i] * bits[:, j]
# energy[k]는 비트열 k의 Ising 해밀토니안 대각 원소입니다.
scale = max(1, int(np.max(np.abs(energy))))

# 4. 균등 중첩, 비용 회로, X 믹서
def probabilities(angles):
    gamma, beta = angles  # 감마는 비용 위상 각도, 베타는 믹서 각도
    state = np.exp(-1j * gamma * energy / scale) / np.sqrt(len(bits))
    for q in range(n):
        flipped = index ^ (1 << q)
        state = np.cos(beta) * state - 1j * np.sin(beta) * state[flipped]
    return np.abs(state)**2

# 5. 고전 최적화기로 각도 찾기
def expectation(angles):
    return float(probabilities(angles) @ energy)

fit = minimize(expectation, [1.0, 0.5], method="COBYLA",
               options={"maxiter": 60})
prob = probabilities(fit.x)
assert np.isclose(prob.sum(), 1.0)

# 6. 모의 측정과 완전탐색 검증
shots = 4096
counts = np.random.default_rng(42).multinomial(shots, prob / prob.sum())
valid = index[feasible]
observed = index[feasible & (counts > 0)]
print(f"대상 {group}, 예산 {budget:,}원")
print(f"기대 에너지 {expectation(fit.x):.4f}")
print(f"실행 가능 해 관측률 {counts[valid].sum() / shots:.1%}")
if len(valid) == 0:
    print("예산 안에서 두 개를 고를 수 없습니다.")
else:
    best_score = int(score[valid].max())
    optimal = valid[score[valid] == best_score]
    print(f"완전탐색 최적값 {best_score}점")
    print(f"최적해 관측률 {counts[optimal].sum() / shots:.1%}")
    if len(observed) == 0:
        print("이번 측정에서 조건을 만족하는 조합이 나오지 않았습니다.")
    else:
        selected = observed[np.argmax(score[observed])]
        chosen = [names[i] for i in range(n) if bits[selected, i] == 1]
        assert count[selected] == 2 and spending[selected] <= budget
        print("선택 비트", "".join(map(str, bits[selected])))
        print("선택 항목", ", ".join(chosen))
        print(f"선호도 {score[selected] - bonus[selected]}점, 궁합 {bonus[selected]}점")
        print(f"선택 점수 {score[selected]}점, 최적값과 차이 {best_score - score[selected]}점")
        print(f"수량 {count[selected]}개 충족, 비용 {spending[selected]:,}원으로 예산 충족")

대상 성인, 예산 12,000원
기대 에너지 60.0116
실행 가능 해 관측률 31.8%
완전탐색 최적값 13점
최적해 관측률 1.2%
선택 비트 11000000
선택 항목 소고기, 버섯
선호도 7점, 궁합 6점
선택 점수 13점, 최적값과 차이 0점
수량 2개 충족, 비용 12,000원으로 예산 충족
